# DePlag Accuracy Upgrade Notebook

Goal: improve paraphrased-plagiarism detection using a stronger sentence-pair cross-encoder while keeping evaluation honest.

This notebook does not claim 95%+ accuracy in advance. It trains, validates, checks leakage risk, tunes the decision threshold on validation data only, then reports final metrics on a held-out test set.

## Runtime Notes

Use Google Colab with GPU: `Runtime -> Change runtime type -> T4/A100 GPU`.

Recommended first run: `microsoft/deberta-v3-base`, `MAX_TRAIN_EXAMPLES=120000`, `EPOCHS=3`.

For a stronger but slower run: `microsoft/deberta-v3-large`, lower batch size, and use A100 if available.

In [ ]:
# CELL 1 - Install dependencies
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn sentencepiece protobuf

import os, json, random, re, math, time
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi

In [ ]:
# CELL 2 - Optional Google Drive mount
# Uncomment if you want saved models/results copied to Drive.
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT = '/content/DePlag_Accuracy_Upgrade'
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
for d in ['data', 'models', 'reports', 'evaluation']:
    os.makedirs(d, exist_ok=True)
print('Working directory:', os.getcwd())

## Configuration

The main accuracy upgrade is the cross-encoder. Unlike a bi-encoder, it jointly attends over both texts and is usually much stronger for pair classification.

In [ ]:
# CELL 3 - Experiment configuration

MODEL_NAME = 'microsoft/deberta-v3-base'
# Stronger, slower option:
# MODEL_NAME = 'microsoft/deberta-v3-large'

MAX_LENGTH = 192
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
WARMUP_RATIO = 0.08

# Set None for full data. Use a cap for faster first experiments on T4.
MAX_TRAIN_EXAMPLES = 120000
MAX_EVAL_EXAMPLES = None

# Dataset switches. QQP is the biggest boost; PAWS gives adversarial hard negatives.
USE_MRPC = True
USE_PAWS = True
USE_STSB = True
USE_QQP = True
USE_MACHINE_PARAPHRASE = True
USE_LOCAL_RAW_CSV = True
LOCAL_RAW_CSV_PATHS = [
    'data/raw/paraphrase data.csv',
    'data/raw/paraphrased_articles.csv',
    '/content/paraphrase data.csv',
    '/content/paraphrased_articles.csv',
]

print({
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'lr': LR,
    'max_train_examples': MAX_TRAIN_EXAMPLES,
})

In [ ]:
# CELL 4 - Dataset loading

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import train_test_split as sk_train_test_split

def minimal_clean(text):
    if not isinstance(text, str):
        return ''
    return ' '.join(text.strip().split())

def norm_for_dedupe(text):
    text = minimal_clean(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return ' '.join(text.split())

def as_pair_df(rows, source):
    df = pd.DataFrame(rows)
    df['text_a'] = df['text_a'].map(minimal_clean)
    df['text_b'] = df['text_b'].map(minimal_clean)
    df = df[(df.text_a != '') & (df.text_b != '')].copy()
    df['label'] = df['label'].astype(int)
    df['source'] = source
    return df[['text_a', 'text_b', 'label', 'source']]

def dataset_to_pair_df(ds_split, source, default_label=None):
    df = ds_split.to_pandas()
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}
    label_col = next((lower[c] for c in ['label', 'labels', 'is_duplicate', 'duplicate', 'paraphrase'] if c in lower), None)
    pair_candidates = [
        ('text_a', 'text_b'), ('sentence1', 'sentence2'), ('question1', 'question2'),
        ('s1', 's2'), ('source', 'target'), ('original', 'paraphrase'),
        ('original_text', 'paraphrased_text'), ('text', 'paraphrase'),
    ]
    text_a_col, text_b_col = None, None
    for a, b in pair_candidates:
        if a in lower and b in lower:
            text_a_col, text_b_col = lower[a], lower[b]
            break
    if text_a_col is None:
        string_cols = [c for c in cols if df[c].dtype == object]
        if len(string_cols) >= 2:
            text_a_col, text_b_col = string_cols[0], string_cols[1]
    if text_a_col is None:
        raise ValueError(f'Could not infer sentence-pair columns for {source}. Columns: {cols}')
    out = pd.DataFrame({'text_a': df[text_a_col], 'text_b': df[text_b_col]})
    if label_col is not None:
        out['label'] = df[label_col].astype(int)
    elif default_label is not None:
        out['label'] = int(default_label)
    else:
        raise ValueError(f'Could not infer label column for {source}. Columns: {cols}')
    return as_pair_df(out.to_dict('records'), source)

def split_single_df(df, val_size=0.10, test_size=0.10):
    if len(df) < 20 or df['label'].nunique() < 2:
        return df, pd.DataFrame(columns=df.columns), pd.DataFrame(columns=df.columns)
    trainval, test = sk_train_test_split(df, test_size=test_size, stratify=df['label'], random_state=SEED)
    rel_val = val_size / (1.0 - test_size)
    train, val = sk_train_test_split(trainval, test_size=rel_val, stratify=trainval['label'], random_state=SEED)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def load_local_paraphrase_csv(paths):
    frames = []
    for path in paths:
        if not os.path.exists(path):
            continue
        raw = pd.read_csv(path)
        lower = {c.lower(): c for c in raw.columns}
        if {'text_a', 'text_b', 'label'}.issubset(lower):
            frames.append(as_pair_df(raw.rename(columns={lower['text_a']: 'text_a', lower['text_b']: 'text_b', lower['label']: 'label'}).to_dict('records'), 'local_csv'))
            continue
        paired_cols = [
            ('title', 'paraphrasedtitle'),
            ('abstract', 'paraphrasedabstract'),
            ('introduction', 'paraphraseintroduction'),
        ]
        rows = []
        for a, b in paired_cols:
            if a in lower and b in lower:
                for _, r in raw.iterrows():
                    rows.append({'text_a': r[lower[a]], 'text_b': r[lower[b]], 'label': 1})
                shuffled = raw[lower[b]].sample(frac=1, random_state=SEED).reset_index(drop=True)
                for i, (_, r) in enumerate(raw.iterrows()):
                    rows.append({'text_a': r[lower[a]], 'text_b': shuffled.iloc[i], 'label': 0})
        if rows:
            frames.append(as_pair_df(rows, 'local_csv'))
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_parts, val_parts, test_parts = [], [], []

if USE_MRPC:
    mrpc = load_dataset('glue', 'mrpc')
    def mrpc_rows(split):
        return [{'text_a': x['sentence1'], 'text_b': x['sentence2'], 'label': x['label']} for x in mrpc[split]]
    train_parts.append(as_pair_df(mrpc_rows('train'), 'mrpc'))
    val_parts.append(as_pair_df(mrpc_rows('validation'), 'mrpc'))

if USE_PAWS:
    paws = load_dataset('paws', 'labeled_final')
    def paws_rows(split):
        return [{'text_a': x['sentence1'], 'text_b': x['sentence2'], 'label': x['label']} for x in paws[split]]
    train_parts.append(as_pair_df(paws_rows('train'), 'paws'))
    val_parts.append(as_pair_df(paws_rows('validation'), 'paws'))
    test_parts.append(as_pair_df(paws_rows('test'), 'paws'))

if USE_QQP:
    qqp = load_dataset('glue', 'qqp')
    def qqp_rows(split):
        rows = []
        for x in qqp[split]:
            if x['question1'] and x['question2']:
                rows.append({'text_a': x['question1'], 'text_b': x['question2'], 'label': x['label']})
        return rows
    train_parts.append(as_pair_df(qqp_rows('train'), 'qqp'))
    val_parts.append(as_pair_df(qqp_rows('validation'), 'qqp'))

if USE_STSB:
    try:
        stsb = load_dataset('mteb/stsbenchmark-sts')
        score_key, s1_key, s2_key = 'score', 'sentence1', 'sentence2'
    except Exception:
        stsb = load_dataset('stsb_multi_mt', 'en')
        score_key, s1_key, s2_key = 'similarity_score', 'sentence1', 'sentence2'
    def stsb_rows(split):
        rows = []
        for x in stsb[split]:
            rows.append({
                'text_a': x[s1_key],
                'text_b': x[s2_key],
                'label': 1 if float(x[score_key]) >= 3.5 else 0,
            })
        return rows
    train_parts.append(as_pair_df(stsb_rows('train'), 'stsb'))
    val_parts.append(as_pair_df(stsb_rows('validation'), 'stsb'))
    test_parts.append(as_pair_df(stsb_rows('test'), 'stsb'))

if USE_MACHINE_PARAPHRASE:
    mp = load_dataset('jpwahle/machine-paraphrase-dataset')
    print('Machine-paraphrase splits:', mp)
    if 'train' in mp and ('validation' in mp or 'test' in mp):
        train_parts.append(dataset_to_pair_df(mp['train'], 'machine_paraphrase', default_label=1))
        if 'validation' in mp:
            val_parts.append(dataset_to_pair_df(mp['validation'], 'machine_paraphrase', default_label=1))
        if 'test' in mp:
            test_parts.append(dataset_to_pair_df(mp['test'], 'machine_paraphrase', default_label=1))
    else:
        split_name = 'train' if 'train' in mp else list(mp.keys())[0]
        mp_df = dataset_to_pair_df(mp[split_name], 'machine_paraphrase', default_label=1)
        tr, va, te = split_single_df(mp_df)
        train_parts.append(tr)
        if len(va): val_parts.append(va)
        if len(te): test_parts.append(te)

if USE_LOCAL_RAW_CSV:
    local_df = load_local_paraphrase_csv(LOCAL_RAW_CSV_PATHS)
    if local_df is None:
        print('No local raw CSV found. In Colab, upload it to data/raw/paraphrase data.csv or data/raw/paraphrased_articles.csv')
    else:
        tr, va, te = split_single_df(local_df)
        train_parts.append(tr)
        if len(va): val_parts.append(va)
        if len(te): test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print('Raw sizes:', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print('Train sources:')
print(train_df.source.value_counts())
print('Val sources:')
print(val_df.source.value_counts())
print('Test sources:')
print(test_df.source.value_counts())

In [ ]:
# CELL 5 - Deduplication and leakage checks

def add_pair_key(df):
    a = df['text_a'].map(norm_for_dedupe)
    b = df['text_b'].map(norm_for_dedupe)
    left = np.minimum(a, b)
    right = np.maximum(a, b)
    df = df.copy()
    df['pair_key'] = left + ' ||| ' + right
    return df

train_df = add_pair_key(train_df)
val_df = add_pair_key(val_df)
test_df = add_pair_key(test_df)

before = {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)}
train_df = train_df.drop_duplicates('pair_key').reset_index(drop=True)
val_df = val_df.drop_duplicates('pair_key').reset_index(drop=True)
test_df = test_df.drop_duplicates('pair_key').reset_index(drop=True)

train_keys = set(train_df.pair_key)
val_keys = set(val_df.pair_key)
test_keys = set(test_df.pair_key)

leak_val = len(train_keys & val_keys)
leak_test = len((train_keys | val_keys) & test_keys)
print('Before dedupe:', before)
print('After dedupe:', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print('Exact pair leakage train-val:', leak_val)
print('Exact pair leakage train/val-test:', leak_test)

# Remove exact overlaps from validation/test to protect evaluation.
val_df = val_df[~val_df.pair_key.isin(train_keys)].reset_index(drop=True)
test_df = test_df[~test_df.pair_key.isin(train_keys | set(val_df.pair_key))].reset_index(drop=True)

if MAX_TRAIN_EXAMPLES is not None and len(train_df) > MAX_TRAIN_EXAMPLES:
    train_df = train_df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), MAX_TRAIN_EXAMPLES // 2), random_state=SEED)
    ).sample(frac=1, random_state=SEED).reset_index(drop=True)

if MAX_EVAL_EXAMPLES is not None:
    val_df = val_df.sample(min(len(val_df), MAX_EVAL_EXAMPLES), random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(min(len(test_df), MAX_EVAL_EXAMPLES), random_state=SEED).reset_index(drop=True)

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print('\n', name, len(df))
    print(df.label.value_counts(normalize=True).rename({0: 'not_plagiarism', 1: 'plagiarism'}))
    print(df.source.value_counts())

train_df.drop(columns=['pair_key']).to_csv('data/train_pairs.csv', index=False)
val_df.drop(columns=['pair_key']).to_csv('data/val_pairs.csv', index=False)
test_df.drop(columns=['pair_key']).to_csv('data/test_pairs.csv', index=False)

In [ ]:
# CELL 6 - Tokenization

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_dataset(df):
    return Dataset.from_pandas(df[['text_a', 'text_b', 'label', 'source']].reset_index(drop=True), preserve_index=False)

raw_ds = DatasetDict({
    'train': to_dataset(train_df),
    'validation': to_dataset(val_df),
    'test': to_dataset(test_df),
})

def tokenize_batch(batch):
    return tokenizer(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LENGTH)

encoded = raw_ds.map(tokenize_batch, batched=True, remove_columns=['text_a', 'text_b', 'source'])
encoded = encoded.rename_column('label', 'labels')
print(encoded)

In [ ]:
# CELL 7 - Metrics and threshold helper

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef, precision_recall_curve, classification_report, confusion_matrix

def find_best_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-9, None)
    idx = int(np.argmax(f1))
    return float(thresholds[idx]), float(f1[idx])

def metric_dict(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    out = {
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'mcc': float(matthews_corrcoef(y_true, pred)),
        'threshold': float(threshold),
    }
    if len(set(y_true)) == 2:
        out['roc_auc'] = float(roc_auc_score(y_true, proba))
    return out

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    return metric_dict(labels, proba, threshold=0.5)

In [ ]:
# CELL 8 - Train cross-encoder

from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

steps_per_epoch = max(1, math.ceil(len(encoded['train']) / TRAIN_BATCH_SIZE))
eval_steps = max(100, steps_per_epoch // 2)

args = TrainingArguments(
    output_dir='models/cross_encoder',
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=int(steps_per_epoch * EPOCHS * WARMUP_RATIO),
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=eval_steps,
    save_steps=eval_steps,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['validation'],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
trainer.save_model('models/best_cross_encoder')
tokenizer.save_pretrained('models/best_cross_encoder')

In [ ]:
# CELL 9 - Validation threshold tuning and final test evaluation

def predict_proba(split_name):
    pred = trainer.predict(encoded[split_name])
    logits = pred.predictions
    proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    labels = np.asarray(pred.label_ids)
    return labels, proba

y_val, val_proba = predict_proba('validation')
best_threshold, best_val_f1 = find_best_threshold(y_val, val_proba)
val_metrics = metric_dict(y_val, val_proba, best_threshold)

y_test, test_proba = predict_proba('test')
test_metrics = metric_dict(y_test, test_proba, best_threshold)
test_pred = (test_proba >= best_threshold).astype(int)

print('Best validation threshold:', best_threshold)
print('Validation metrics:', json.dumps(val_metrics, indent=2))
print('Held-out test metrics:', json.dumps(test_metrics, indent=2))
print('\nClassification report:')
print(classification_report(y_test, test_pred, target_names=['Not Plagiarism', 'Plagiarism']))

results = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'weight_decay': WEIGHT_DECAY,
    'train_examples': len(train_df),
    'validation_examples': len(val_df),
    'test_examples': len(test_df),
    'validation': val_metrics,
    'test': test_metrics,
    'confusion_matrix': confusion_matrix(y_test, test_pred).tolist(),
    'test_classification_report': classification_report(y_test, test_pred, target_names=['Not Plagiarism', 'Plagiarism'], output_dict=True),
}

with open('reports/results.json', 'w') as f:
    json.dump(results, f, indent=2)

with open('reports/classifier_report.txt', 'w') as f:
    f.write(json.dumps(results, indent=2))

print('Saved reports/results.json and reports/classifier_report.txt')

In [ ]:
# CELL 10 - Confusion matrix figure

import matplotlib.pyplot as plt
import seaborn as sns

cm = np.array(results['confusion_matrix'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Plagiarism', 'Plagiarism'],
            yticklabels=['Not Plagiarism', 'Plagiarism'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f"Held-Out Test Confusion Matrix\nAcc={test_metrics['accuracy']:.3f} F1={test_metrics['f1']:.3f} AUC={test_metrics.get('roc_auc', 0):.3f}")
plt.tight_layout()
plt.savefig('evaluation/confusion_matrix_cross_encoder.png', dpi=300)
plt.show()
print('Saved evaluation/confusion_matrix_cross_encoder.png')

In [ ]:
# CELL 11 - Error analysis sample

analysis_df = test_df[['text_a', 'text_b', 'label', 'source']].copy().reset_index(drop=True)
analysis_df['proba'] = test_proba
analysis_df['pred'] = test_pred
analysis_df['error_type'] = np.where(
    (analysis_df.label == 0) & (analysis_df.pred == 1), 'false_positive',
    np.where((analysis_df.label == 1) & (analysis_df.pred == 0), 'false_negative', 'correct')
)
analysis_df.to_csv('reports/test_predictions.csv', index=False)

print('False positives:')
display(analysis_df[analysis_df.error_type == 'false_positive'].sort_values('proba', ascending=False).head(10))
print('False negatives:')
display(analysis_df[analysis_df.error_type == 'false_negative'].sort_values('proba', ascending=True).head(10))
print('Saved reports/test_predictions.csv')

## Optional Accuracy Extensions

If the first run does not approach 95%+, try these in order:

1. Increase `MAX_TRAIN_EXAMPLES` or set it to `None`.
2. Use `microsoft/deberta-v3-large` on A100.
3. Increase `EPOCHS` to 4 or 5 with early stopping.
4. Raise `MAX_LENGTH` to 256 for longer academic sentences.
5. Add more high-quality paraphrase/plagiarism datasets after checking license and label semantics.
6. Train a small ensemble using cross-encoder probability, SBERT cosine, Jaccard overlap, and length-ratio features.
7. Run official split evaluation per dataset and report per-dataset metrics, not only combined metrics.

In [ ]:
# CELL 12 - Optional Drive export

# Uncomment after mounting Drive in Cell 2.
# !mkdir -p /content/drive/MyDrive/DePlag_Accuracy_Upgrade
# !cp -r models reports evaluation data /content/drive/MyDrive/DePlag_Accuracy_Upgrade/
# print('Copied artifacts to Google Drive')